# EDA — Olist Customer Reviews

**Data:** 2026-11-06

Exploração inicial do dataset de reviews da Olist. Objetivo é entender a distribuição de scores, volume temporal e características dos textos antes de indexar tudo no ChromaDB.

In [ ]:
import sys
from pathlib import Path

# notebooks/ está um nível abaixo do projeto — adiciona a raiz ao path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style="whitegrid")
print("imports ok")

In [ ]:
from src.ingestion import load_reviews, summarize_reviews

In [ ]:
# carrega o CSV — espera que data/raw/olist_order_reviews_dataset.csv exista
DATA_PATH = Path.cwd().parent / "data" / "raw" / "olist_order_reviews_dataset.csv"

df_raw = pd.read_csv(DATA_PATH)
df_raw.head()

In [ ]:
print("Shape:", df_raw.shape)
print()
print(df_raw.dtypes)

In [ ]:
print("Nulos por coluna:")
print(df_raw.isnull().sum())

**Nota:** Muitos reviews não têm `review_comment_message` preenchido. Esses registros são descartados no `load_reviews()` porque não há texto para indexar. O pipeline usa apenas reviews com texto.

In [ ]:
# distribuição de review_score
score_counts = df_raw["review_score"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(7, 4))
score_counts.plot(kind="bar", ax=ax, color="steelblue", edgecolor="white")
ax.set_title("Distribuição de review_score")
ax.set_xlabel("Score")
ax.set_ylabel("Quantidade")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.xticks(rotation=0)
plt.tight_layout()

img_dir = Path.cwd().parent / "images"
img_dir.mkdir(exist_ok=True)
plt.savefig(img_dir / "eda_score_distribution.png", dpi=150)
plt.show()
print("salvo em images/eda_score_distribution.png")

In [ ]:
# volume por mês-ano
df_raw["review_creation_date"] = pd.to_datetime(df_raw["review_creation_date"])
df_raw["month_year"] = df_raw["review_creation_date"].dt.to_period("M")

monthly = df_raw.groupby("month_year").size().reset_index(name="count")
monthly["month_year_str"] = monthly["month_year"].astype(str)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(monthly["month_year_str"], monthly["count"], marker="o", linewidth=1.5)
ax.set_title("Volume de reviews por mês")
ax.set_xlabel("Mês")
ax.set_ylabel("Quantidade")
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# comprimento médio do texto por score
df_com_texto = df_raw.dropna(subset=["review_comment_message"]).copy()
df_com_texto["text_len"] = df_com_texto["review_comment_message"].str.len()

avg_len = df_com_texto.groupby("review_score")["text_len"].mean().round(1)
print("Comprimento médio do texto por score:")
print(avg_len)

In [ ]:
# exemplos por categoria de score
for score in [1, 3, 5]:
    sample = (
        df_com_texto[df_com_texto["review_score"] == score]["review_comment_message"]
        .dropna()
        .sample(n=2, random_state=42)
    )
    print(f"\n--- Score {score} ---")
    for i, text in enumerate(sample, 1):
        print(f"[{i}] {text[:200]}")

## TODO

- [ ] Analisar distribuição de comprimento de texto (histograma)
- [ ] Ver se há correlação entre score e presença de título (`review_comment_title`)
- [ ] Verificar reviews duplicados (mesmo `order_id`, scores diferentes)
- [ ] Explorar sazonalidade — Black Friday aparece em novembro?
- [ ] Checar encoding — acentos estão OK?
- [ ] Testar `load_reviews()` e comparar com df_raw (quantos ficam após filtro de texto vazio)